This notebook performs:
1. Selection of the item subgroup - this subgroup will serve us through the entire pipeline.
2. Data augmentation - training sessions with labels from the item subgroup are used to augment additional data. We performed a simple augmentation, each of these session (about 100k) has a 50% chance of the views to be removed. Sessions have at least 1 view. This additional data will be added to the training data. One problem that may arise is that by removing views we change the time features of the session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os.path
from datetime import datetime
import matplotlib.pyplot as plt
import random

base_path = '/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset'
original_data = os.path.join(base_path, 'original_data')
processed_data = os.path.join(base_path, 'processed_data')

In [ ]:
item_features = pd.read_csv(os.path.join(original_data, "item_features.csv"))
test_full_sessions = pd.read_csv(os.path.join(original_data, "test_full_sessions.csv"))
test_full_purchases = pd.read_csv(os.path.join(original_data, "test_full_purchases.csv"))
train_sessions = pd.read_csv(os.path.join(original_data, "train_sessions.csv"))
train_purchases = pd.read_csv(os.path.join(original_data, "train_purchases.csv"))

In [ ]:
# Randomly selecting Category and Feature value for defining the item subgroup
random.seed(14)

item_percentile = 0
train_percentile = 0
test_percentile = 0
n_feature_vals = 0

total_unique_items = item_features['item_id'].nunique()
category_value_counts = item_features.groupby(['feature_category_id', 'feature_value_id'])['item_id'].nunique()

selected_category_value_set = set()
unique_item_ids = set()
while item_percentile < 0.10:
    random_index = random.randint(0, len(category_value_counts) - 1)
    random_category_value = category_value_counts.index[random_index]
    selected_category_value_set.add(random_category_value)
    unique_item_ids.update(item_features[item_features.set_index(['feature_category_id', 'feature_value_id']).index == random_category_value]['item_id'].unique())
    item_percentile = len(unique_item_ids) / total_unique_items

train_items = pd.concat([train_sessions['item_id'], train_purchases['item_id']])
train_count = train_items.isin(unique_item_ids).sum()
train_percentage = train_count / len(train_items)

test_items = pd.concat([test_full_sessions['item_id'], test_full_purchases['item_id']])
test_count = test_items.isin(unique_item_ids).sum()
test_percentage = test_count / len(test_items)


print(f"Item Percentage: {item_percentile}")
print(f'Train Percentage: {train_percentage}')
print(f'Test Percentage: {test_percentage}')
print(f"Selected (Category, Feature Values) : {selected_category_value_set}")

Item Percentage: 0.12519522181419104
Train Percentage: 0.09395924663377334
Test Percentage: 0.07620633570500811
Selected (Category, Feature Values) : {(62, 15), (67, 382), (8, 636), (56, 295)}


In [ ]:
# Selecting sessions from training from wich we will augment additional data
special_session_ids_list = train_purchases["session_id"][train_purchases["item_id"].isin(unique_item_ids)].tolist()
print(f"Percent of train sessions with label from item subgroup: {len(special_session_ids_list)/len(train_purchases)}")

Percent of train sessions with label from item subgroup: 0.101096


In [ ]:
aug_train_sessions = train_sessions[train_sessions["session_id"].isin(special_session_ids_list)].reset_index(drop=True)
aug_train_purchases = train_purchases[train_purchases["session_id"].isin(special_session_ids_list)].reset_index(drop=True)

In [ ]:
def augment_session(group):
    if len(group) > 1:                                       # Only augment if there's more than one view in the session
        drop_mask = np.random.rand(len(group)) < 0.5         # Randomly mark views to be dropped, with a probability of 0.5
        if drop_mask.all():                                  # Ensure at least one view is kept
            drop_mask[np.random.choice(len(group))] = False  # Keep a random one
        return group[~drop_mask]
    else:
        return group

# Apply the augmentation function to each group of session_id's and concatenate the results
aug_train_sessions = aug_train_sessions.groupby('session_id').apply(augment_session)

# Reset the index since 'apply' returns a multi-indexed DF
aug_train_sessions.reset_index(drop=True, inplace=True)
print(f"Mean train session length: {train_sessions.groupby('session_id').size().mean()}")
print(f"Mean augmented session length: {aug_train_sessions.groupby('session_id').size().mean()}")

Mean train session length: 4.74382
Mean augmented session length: 2.366898789269605


In [ ]:
# Checking how many session stayed identical
original_counts = train_sessions.groupby('session_id').size()
aug_counts = aug_train_sessions.groupby('session_id').size()
original_counts_train_common = original_counts[original_counts.index.isin(special_session_ids_list)]
original_counts_aug_common = aug_counts[aug_counts.index.isin(special_session_ids_list)]
identical_sessions_by_size = (original_counts_train_common == original_counts_aug_common).sum()
print(f"Number of identical sessions : {identical_sessions_by_size}")

Number of identicle sessions : 42058


In [ ]:
# ~42k of 101k augmented sessions are identicle so we need to filter them out along with the aug purchases
# Finally we add 59k new augmented sessions
different_sessions = aug_counts[~(aug_counts == original_counts.reindex(aug_counts.index))]
filtered_aug_sessions = aug_train_sessions[aug_train_sessions['session_id'].isin(different_sessions.index)]
filtered_aug_train_purchases = aug_train_purchases[aug_train_purchases['session_id'].isin(filtered_aug_sessions['session_id'])]

In [ ]:
# Save augmented train session and purchases
filtered_aug_sessions.to_csv(os.path.join(original_data, "aug_train_sessions.csv"), index=False)
filtered_aug_train_purchases.to_csv(os.path.join(original_data, "aug_train_purchases.csv"), index=False)

In [ ]:
# Save selected Category, Feature id for global use
subgroup_category_feature_df = pd.DataFrame(list(selected_category_value_set), columns=['feature_category_id', 'feature_value_id'])
subgroup_category_feature_df.to_csv(os.path.join(original_data, "subgroup_cat_feat.csv"), index=False)